# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Fraud-Ops War Room

### Scenario

You've joined the **Fraud Operations** team at **PayKart**, a card-payments company. Last month's numbers looked wrong and
the Head of Fraud Ops has called a review. You have 30 days of transactions with the automated detector's score and the
investigators' confirmed labels (`unit3_ppt7_card_transactions.csv`). Their requests:

1. *"I don't want a list of alerts. Show me the whole month first — is anything unusual going on?"*
2. *"The investigators keep saying 'night-time, far-from-home' transactions are risky. Show me, across ALL our views at
   once, what those transactions look like."*
3. *"Engineering wants to build the interactive filter tool on our full history: 50 million transactions. Will a slider
   still feel instant?"*
4. *"The detector threshold is 0.5 and the team is drowning in alerts. We can review at most **150 alerts a month**. Where
   should the threshold be?"*
5. *"Give the investigators a tool where they can poke at the flagged cases themselves."*
6. *"Someone says it was a fraud ring in one city. Prove it — or kill the idea."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer in your own
words. Interactive (plotly) figures support zoom, pan, hover, selection and clicking.

## Part 0 — Setup (given)

Run this cell as-is. It loads `tx` and adds `night` (hour < 6) and a readable `label` column.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
# If plotly figures come out blank: classic Jupyter -> pio.renderers.default = "notebook";
# Google Colab -> "colab"; VS Code / JupyterLab work out of the box.
import plotly.io as pio

tx = pd.read_csv("unit3_ppt7_card_transactions.csv", parse_dates=["timestamp"])
tx["night"] = tx.hour < 6
tx["label"] = np.where(tx.is_fraud == 1, "confirmed fraud", "legitimate")

print(f"{len(tx):,} transactions | {tx.is_fraud.sum()} confirmed fraud | days {tx.day.min()}-{tx.day.max()}")
tx.head()

## Task 1 — Overview First (Request #1)

**Real-world usage:** Fraud dashboards start from a whole-period overview before any alert list.

**Why this technique:** Shneiderman's mantra — **overview first**, then zoom and filter, then details (Slide 30); a
**reordered** matrix makes patterns pop out (Slide 39).

**TODO:**
1. Plot daily transaction counts with daily confirmed fraud on a second axis.
2. Build a city × day matrix of confirmed-fraud **counts**, reorder rows and columns by their totals, and show it as a
   heatmap with labelled axes.
3. Print the (city, day) cells with the most fraud.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7))
# TODO 1
daily_n = tx.groupby("day").size()
daily_fraud = tx.groupby("day").is_fraud.sum()
axes[0].plot(daily_n.index, daily_n.values, color="grey", lw=2, label="all transactions")
axes[0].set_xlabel("Day of month"); axes[0].set_ylabel("Transactions per day")
ax2 = axes[0].twinx(); ax2.spines["right"].set_visible(True)
ax2.bar(daily_fraud.index, daily_fraud.values, color="red", alpha=0.6, label="confirmed fraud")
ax2.set_ylabel("Confirmed fraud per day", color="red")
axes[0].set_title("Daily volume is steady, but confirmed fraud spikes on a few days")

# TODO 2
heat = tx.pivot_table(index="city", columns="day", values="is_fraud", aggfunc="sum", fill_value=0)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index,      # rows ordered by city total
                heat.sum(axis=0).sort_values(ascending=False).index]      # columns ordered by day total
im = axes[1].imshow(heat.values, cmap="Reds", aspect="auto")
axes[1].set_xticks(range(heat.shape[1])); axes[1].set_xticklabels(heat.columns)
axes[1].set_yticks(range(heat.shape[0])); axes[1].set_yticklabels(heat.index)
axes[1].set_xlabel("Day (reordered by total fraud)"); axes[1].set_ylabel("City (reordered by total fraud)")
axes[1].set_title("Confirmed fraud count by city and day (reordered matrix)")
plt.colorbar(im, ax=axes[1], label="Fraud count")

plt.tight_layout(); plt.show()

# TODO 3
print(heat.stack().sort_values(ascending=False).head(5))

**Reflection:** What did the overview show that an alert list sorted by detector score would *not* have shown? Name the
lecture concept (Slide 38 or 39) that describes how you found it.

_Your answer:_ The overview showed that fraud is not spread evenly. Confirmed fraud is packed into one city (Pune) on days 18, 19 and 20 (17, 13 and 10 cases), while daily transaction volume looks normal on those days. An alert list sorted by detector score is just a list of single transactions, with no place or day attached, so a cluster in space and time would be buried in it. I found it by aggregating to a city by day matrix and reordering the rows and columns by their totals (the reorderable matrix, Slide 39), so the high cells move together into one corner, after starting with the whole month (overview first, Slide 30).


## Task 2 — Linked Views for "Night-Time, Far-From-Home" (Request #2)

**Real-world usage:** Investigators validate rules of thumb by brushing them across a coordinated dashboard.

**Why this technique:** **Cross-view brushing / linked highlighting** reveals how a selection in one view is distributed
in all the others (Slides 8–12).

**TODO:**
1. Define the brush: `night == True` **and** `distance_from_home_km > 100`.
2. Draw four coordinated views with the brushed items in red over all items in grey: (a) hour vs. distance scatter
   (log y), (b) merchant category counts, (c) city counts, (d) amount histogram (log10).
3. Print: number brushed, fraud rate inside the brush vs. overall, and the share of *all* fraud the brush captures.

In [ ]:
# TODO 1
brush = (tx.night == True) & (tx.distance_from_home_km > 100)

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
# TODO 2
# (a) hour vs distance (log y)
axes[0].scatter(tx.hour, tx.distance_from_home_km, s=6, color="lightgrey")
axes[0].scatter(tx.hour[brush], tx.distance_from_home_km[brush], s=10, color="red")
axes[0].set_yscale("log"); axes[0].set_xlabel("Hour of day"); axes[0].set_ylabel("Distance from home (km, log)")
axes[0].set_title("Hour vs distance")

# (b) merchant category counts
cat_all = tx.merchant_category.value_counts()
cat_brush = tx.merchant_category[brush].value_counts().reindex(cat_all.index, fill_value=0)
axes[1].bar(cat_all.index, cat_all.values, color="lightgrey"); axes[1].bar(cat_brush.index, cat_brush.values, color="red")
axes[1].set_title("Merchant category"); axes[1].set_ylabel("Transactions"); axes[1].tick_params(axis="x", rotation=60)

# (c) city counts
city_all = tx.city.value_counts()
city_brush = tx.city[brush].value_counts().reindex(city_all.index, fill_value=0)
axes[2].bar(city_all.index, city_all.values, color="lightgrey"); axes[2].bar(city_brush.index, city_brush.values, color="red")
axes[2].set_title("City"); axes[2].set_ylabel("Transactions"); axes[2].tick_params(axis="x", rotation=60)

# (d) amount histogram (log10)
bins = np.linspace(np.log10(tx.amount_inr.min()), np.log10(tx.amount_inr.max()), 30)
axes[3].hist(np.log10(tx.amount_inr), bins=bins, color="lightgrey")
axes[3].hist(np.log10(tx.amount_inr[brush]), bins=bins, color="red")
axes[3].set_xlabel("log10(amount in INR)"); axes[3].set_ylabel("Transactions"); axes[3].set_title("Amount")

plt.tight_layout(); plt.show()

# TODO 3
rate_brush = tx.is_fraud[brush].mean()
rate_all = tx.is_fraud.mean()
share_captured = tx.is_fraud[brush].sum() / tx.is_fraud.sum()
print(f"Brushed: {brush.sum()} transactions")
print(f"Fraud rate inside the brush: {rate_brush:.1%} | overall: {rate_all:.1%}")
print(f"Share of all fraud captured by the brush: {share_captured:.1%}")

**Reflection:** The brush has a very high fraud *rate* — but what share of all fraud does it *capture*? What does that
tell the investigators about relying on this rule alone, and which linked view hinted at where the rest of the fraud is?

_Your answer:_ The brush has a fraud rate of about 87% (against about 2.2% overall), but it only captures about 44% of all fraud (53 of 120 cases). More than half of the fraud (67 cases) happens outside the brush, spread over all hours of the day (not just at night), so the rule on its own would miss most of it. Investigators should treat the rule as one useful signal, not as the definition of fraud. The linked city view hints at where the rest is: Pune is the largest city in the brushed counts, and Pune also holds the most fraud outside the brush (27 cases), which points back to the Pune cluster found in Task 1.


## Task 3 — Will the Slider Still Feel Instant? (Request #3)

**Real-world usage:** Interactive tools are sized against latency budgets before they're built.

**Why this technique:** **Dynamic queries** need feedback within ~0.1 s (Slides 15–16).

**TODO:**
1. Build a 1-million-row table by repeating `tx`, and measure the time of the filter
   `(detector_score >= 0.6) & (amount_inr > 3000)` (take the best of 5 runs).
2. Compute the per-item cost (µs) and the maximum number of rows filterable within 0.1 s on this machine.
3. Estimate the response time for 50,000,000 rows and the % of rows that would need to be skipped (pre-indexed /
   pre-aggregated) to meet the 0.1 s budget.

In [ ]:
big = pd.concat([tx] * 182, ignore_index=True)

# TODO 1
times = []
for _ in range(5):
    t0 = time.perf_counter()
    result = big[(big.detector_score >= 0.6) & (big.amount_inr > 3000)]
    times.append(time.perf_counter() - t0)
best_s = min(times)

# TODO 2
per_item_us = best_s / len(big) * 1e6
max_rows = 0.1 / best_s * len(big)

# TODO 3
t_50m = per_item_us * 50_000_000 / 1e6
skip_pct = max(0.0, 100 * (1 - 0.1 / t_50m))

print(f"{len(big):,} rows in {best_s*1000:.1f} ms -> {per_item_us:.4f} us/row -> max {max_rows:,.0f} rows in 0.1 s")
print(f"50M rows -> {t_50m:.2f} s; must skip {skip_pct:.1f}% of rows to stay under 0.1 s")

**Reflection:** Besides indexing, name one **reduce** technique from the lecture (Slide 5 / 14) that would let the slider
stay instant on 50M rows, and what the analyst would lose by using it.

_Your answer:_ Aggregation (or binning) would keep the slider instant: precompute counts per bin (for example per hour, city and score band) so that moving the slider only adds up a few hundred bin counts instead of scanning 50 million rows. Sampling is another reduce technique, where the slider filters a small random sample. What the analyst loses is detail. With aggregation, individual transactions are no longer visible without a separate drill down, and with sampling, rare items such as the few fraud cases can be missing or counted wrongly, which is a serious problem when fraud is only about 2% of the data.


## Task 4 — Steer the Threshold to the Team's Capacity (Request #4)

**Real-world usage:** Alert thresholds are set against investigator capacity, not just statistical optimality.

**Why this technique:** **Human-in-the-loop** steering of a model parameter (Slide 37) evaluated with **precision, recall,
F1** (Slides 40–41).

**TODO:**
1. For thresholds 0.05, 0.10, …, 0.95 compute alerts (flagged count), TP, FP, FN, precision, recall and F1.
2. Find the threshold with the **highest recall** among those producing **≤ 150 alerts**. Also find the best-F1 threshold.
3. Plot the precision–recall curve and mark the current threshold (0.5), the capacity-constrained choice and the best-F1
   choice.
4. Print a one-line recommendation.

In [ ]:
rows = []
# TODO 1
for t in np.round(np.arange(0.05, 1.0, 0.05), 2):
    flagged_t = tx.detector_score >= t
    tp = int((flagged_t & (tx.is_fraud == 1)).sum())
    fp = int((flagged_t & (tx.is_fraud == 0)).sum())
    fn = int(((~flagged_t) & (tx.is_fraud == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    rows.append({"threshold": t, "alerts": int(flagged_t.sum()), "TP": tp, "FP": fp, "FN": fn,
                 "precision": precision, "recall": recall, "F1": f1})

pr = pd.DataFrame(rows)

# TODO 2
capacity_choice = pr[pr.alerts <= 150].sort_values(["recall", "precision"], ascending=False).iloc[0]
best_f1 = pr.loc[pr.F1.idxmax()]

# TODO 3
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pr.recall, pr.precision, "-o", color="grey", ms=4)
current = pr[pr.threshold == 0.5].iloc[0]
for row, name, color in [(current, "current 0.5", "tab:blue"),
                         (capacity_choice, "capacity choice", "red"),
                         (best_f1, "best F1", "green")]:
    ax.plot(row.recall, row.precision, "o", ms=11, color=color, label=f"{name} (t={row.threshold:.2f}, {int(row.alerts)} alerts)")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision and recall as the threshold moves")
ax.legend()
plt.tight_layout(); plt.show()

# TODO 4
print(f"Recommendation: set the threshold to {capacity_choice.threshold:.2f}. It gives {int(capacity_choice.alerts)} alerts "
      f"(within the 150 limit) with precision {capacity_choice.precision:.2f} and recall {capacity_choice.recall:.2f}, "
      f"compared with {int(current.alerts)} alerts at 0.5.")

**Reflection:** The capacity-constrained threshold and the best-F1 threshold may differ. Who should make that call — the
model or the Head of Fraud Ops — and why does visual analytics insist on keeping a human in the loop here (Slides 35–37)?

_Your answer:_ The Head of Fraud Ops should make that call, with the model supplying the numbers. The model can only optimise a score such as F1, which treats a missed fraud and a false alert as equally bad. The business knows things the model does not: how much a missed fraud costs, how much an investigator hour costs, and that the team can review only 150 alerts. Visual analytics keeps the human in the loop (Slides 35 to 37) because a person can move the threshold, see the precision and recall curve change straight away and judge the trade-off, which a single automatic number hides. The person is also accountable for the decision and can notice things the model cannot, such as a new attack pattern.


## Task 5 — An Interactive Tool for the Investigators (Request #5)

**Real-world usage:** Investigators triage flagged cases themselves with zoom, filter and hover-for-details.

**Why this technique:** **Details-on-demand / tooltips** (Slides 22–23), **dynamic queries** (Slide 15) and **drill-down**
(Slide 24) — Shneiderman's mantra as an actual tool.

**TODO:**
1. Interactive scatter (plotly) of all transactions with `detector_score >= 0.3`: x = timestamp, y = amount (log),
   colour = `label`, with a tooltip showing txn_id, customer, city, category, distance and score. Set `dragmode="select"`.
2. Interactive **sunburst** for drill-down: city → merchant category, sized by transaction count, coloured by fraud rate.

In [ ]:
# TODO 1
flagged = tx[tx.detector_score >= 0.3]
fig1 = px.scatter(
    flagged, x="timestamp", y="amount_inr", color="label", log_y=True,
    color_discrete_map={"confirmed fraud": "red", "legitimate": "lightgrey"},
    hover_name="txn_id",
    hover_data={"customer_id": True, "city": True, "merchant_category": True,
                "distance_from_home_km": True, "detector_score": ":.2f",
                "timestamp": False, "amount_inr": ":,.0f"},
    title="Flagged transactions (score 0.3 or higher): hover for details, drag to select")
fig1.update_layout(dragmode="select")
fig1.show()

# TODO 2
g = tx.groupby(["city", "merchant_category"]).agg(n=("is_fraud", "size"), fraud=("is_fraud", "sum")).reset_index()
g["fraud_rate"] = g.fraud / g.n
fig2 = px.sunburst(g, path=["city", "merchant_category"], values="n", color="fraud_rate",
                   color_continuous_scale="Reds", title="Drill down: city, then merchant category (colour = fraud rate)")
fig2.show()

**Reflection:** Tooltips are great for one case at a time. Why would they be a *frustrating* way for an investigator to get
an overview of the 400 flagged cases (Slide 23), and which view from this notebook fixes that?

_Your answer:_ A tooltip shows only one transaction at a time and only while the mouse is on it, so to understand 400 flagged cases an investigator would have to hover over each one and remember what the others said. That is slow, tiring and easy to get wrong, and no pattern across cases can appear. Details on demand is meant to come after an overview, not to replace it. The overview views from this notebook fix this: the city by day heatmap and the daily fraud chart (Task 1) show the whole set at once and point to Pune on days 18 to 20, and the sunburst (Task 5) summarises cities and categories, so the investigator can then hover over only the interesting cases.


## Task 6 — Prove or Kill the "Fraud Ring" Hypothesis (Request #6)

**Real-world usage:** Before escalating to law enforcement, the team must rule out simpler explanations.

**Why this technique:** **Hypothesis testing** with visual + statistical confirmation and an explicit **counter-example
search** to guard against confirmation bias (Slides 42–43).

**TODO:**
1. State the hypothesis precisely using what you found in Task 1 (city and day range).
2. Compute the fraud rate inside vs. outside that city–day window and a chi-square p-value
   (`scipy.stats.chi2_contingency` on `pd.crosstab(in_window, tx.is_fraud)`).
3. Counter-example searches: fraud rate for (a) the same city on other days, (b) other cities on the same days.
4. Visual confirmation: a bar chart of the four groups (window / same city other days / other cities same days / rest).
5. Print your verdict.

In [ ]:
from scipy.stats import chi2_contingency

# TODO 1: hypothesis as a boolean mask
# Hypothesis: a concentrated fraud attack in Pune on days 18 to 20 (the hot cells in the Task 1 matrix).
in_day_range = tx.day.between(18, 20)
in_pune = tx.city == "Pune"
in_window = in_pune & in_day_range

# TODO 2
rate_in = tx.is_fraud[in_window].mean()
rate_out = tx.is_fraud[~in_window].mean()
chi2, p_value, _, _ = chi2_contingency(pd.crosstab(in_window, tx.is_fraud))
print(f"Inside the window (Pune, days 18-20): {in_window.sum()} transactions, fraud rate {rate_in:.1%}")
print(f"Outside the window: fraud rate {rate_out:.1%} | chi-square p-value = {p_value:.2e}")

# TODO 3
same_city_other_days = in_pune & ~in_day_range
other_cities_same_days = ~in_pune & in_day_range
rest = ~in_pune & ~in_day_range
rate_a = tx.is_fraud[same_city_other_days].mean()
rate_b = tx.is_fraud[other_cities_same_days].mean()
rate_rest = tx.is_fraud[rest].mean()
print(f"(a) Pune on other days: {rate_a:.1%} | (b) other cities on days 18-20: {rate_b:.1%} | rest: {rate_rest:.1%}")

# TODO 4
fig, ax = plt.subplots(figsize=(8, 4))
names = ["Pune, days 18-20\n(window)", "Pune, other days", "Other cities,\ndays 18-20", "Rest"]
rates = [rate_in, rate_a, rate_b, rate_rest]
bars = ax.bar(names, [r * 100 for r in rates], color=["red", "lightgrey", "lightgrey", "lightgrey"])
for b, r in zip(bars, rates):
    ax.text(b.get_x() + b.get_width() / 2, r * 100 + 0.5, f"{r:.1%}", ha="center")
ax.set_ylabel("Fraud rate (%)")
ax.set_title("Fraud is concentrated in Pune on days 18 to 20 only")
plt.tight_layout(); plt.show()

# TODO 5
if p_value < 0.001 and rate_in > 5 * max(rate_a, rate_b, rate_rest):
    print("Verdict: SUPPORTED. The fraud rate in the window is far above all three comparison groups, so it is not a Pune "
          "problem in general or a bad few days everywhere. It is a localised burst. The statistics show a burst, not a ring "
          "as such, so check shared merchants and cards next.")
else:
    print("Verdict: NOT SUPPORTED. The window is not clearly different from the comparison groups.")

**Reflection:** Which of your two counter-example searches would have *killed* the hypothesis if it had come out
differently, and what alternative explanation would it have supported instead?

_Your answer:_ Both searches could have killed it. If (a) Pune on other days had shown a similarly high fraud rate (it is only 1.9% against 37.4% in the window), then it would be a long-running problem in Pune, such as a weak merchant or terminal there, and not an event on days 18 to 20. If (b) other cities on days 18 to 20 had also been high (it is only 0.4%), then the cause would be a network wide event on those days, such as a data breach or a detector fault, and not something about Pune. Because both stay low, the burst is specific to Pune and to those three days.


## Task 7 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence note to the Head of Fraud Ops covering the whole notebook. It should:
- State what happened last month in one sentence, with numbers.
- Recommend the alert threshold and justify it with capacity, precision/recall and the human-in-the-loop idea.
- Recommend the **one** interactive view to build first for the investigators, and justify it with at least one lecture
  idea (overview first, linked brushing, dynamic queries with the 0.1 s budget, details-on-demand).
- Name the feedback loop (Slide 51) that should be added so next month's detector is better.

_Your final note here:_ Last month 120 of 5,520 transactions (2.2%) were confirmed fraud, and 40 of them (about a third) were in Pune on days 18 to 20, where the fraud rate was 37.4% against 1.5% everywhere else. I recommend setting the threshold to 0.80, which gives 144 alerts (inside the 150 limit) with recall of 0.70 and precision of 0.58, instead of 260 alerts at 0.5. The best F1 threshold (0.85) is close, but the Head of Fraud Ops should make the final call, because only a person knows the cost of a missed fraud against an hour of investigator time, which is the reason to keep a human in the loop. The first interactive view I would build is the overview with linked brushing, starting from the city by day heatmap and daily fraud chart, because overview first lets the investigators see the Pune burst in seconds, and brushing then shows its transactions in all other views, with tooltips only for details. At 50 million rows the sliders will need pre-aggregated data to stay within the 0.1 s budget. To make next month's detector better I would add the iterative analytical reasoning loop from Slide 51 as a fixed monthly routine: start with the overview (daily chart and city by day matrix), filter and select the suspicious subset (the Pune cluster), drill down into its individual transactions, then revise the hypothesis and start again with a sharper question. The revised hypothesis and the newly confirmed fraud labels from each cycle are then used to adjust the detector and its threshold, so each month starts from what the last cycle learned.
